In [1]:
import pandas as pd
import re
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import time
import random

In [ ]:
# Configuration
TARGET_COUNT = 10000  # Set to 6000 for full run
PAGES_TO_FETCH = 10  # 10 pages from SteamSpy approx = 10,000 AppIDs
OUTPUT_FILE = "steam_games_dataset.xlsx"
REF_DATE = datetime(2026, 3, 22)

def extract_steam_features_full(soup):
    f = {}
    
    # Basic Info
    title_tag = soup.find(id="appHubAppName")
    f['Title'] = title_tag.get_text(strip=True) if title_tag else "Unknown"
    f['Developer'] = soup.find(id="developers_list").get_text(strip=True) if soup.find(id="developers_list") else "N/A"

    # Target & Sentiment
    review_meta = soup.find('meta', itemprop='reviewCount')
    f['target_total_reviews'] = float(review_meta['content']) if review_meta else 0.0
    
    review_row = soup.select_one(".user_reviews_summary_row")
    if review_row and review_row.has_attr('data-tooltip-html'):
        pct_match = re.search(r'(\d+)%', review_row['data-tooltip-html'])
        f['Positive_Review_Pct'] = float(pct_match.group(1)) if pct_match else 0.0

    # Temporal Feature
    rel_date_section = soup.select_one(".release_date .date")
    if rel_date_section:
        date_text = rel_date_section.get_text(strip=True)
        for fmt in ('%d %b, %y', '%d %b, %Y', '%b %d, %Y', '%B %d, %Y'):
            try:
                rel_dt = datetime.strptime(date_text, fmt)
                f['game_age_days'] = float((REF_DATE - rel_dt).days)
                break
            except ValueError:
                f['game_age_days'] = 0.0

    # Languages & OS
    lang_table = soup.find("table", class_="game_language_options")
    f['count_languages'] = float(len(lang_table.find_all("tr")) - 1) if lang_table else 1.0
    
    english_audio = False
    if lang_table:
        for tr in lang_table.find_all("tr"):
            tds = tr.find_all('td')
            if len(tds) >= 3 and "English" in tds[0].text:
                if "✔" in tds[2].text:
                    english_audio = True
                    break
    f['has_audio_english'] = english_audio
    
    platforms = soup.select(".game_area_purchase_platform .platform_img")
    p_list = [p['class'][1] for p in platforms if len(p['class']) > 1]
    f['count_os_supported'] = float(len(set(p_list)))
    f['is_on_linux'] = 'linux' in p_list
    f['is_on_mac'] = 'mac' in p_list

    # Technical Specs
    sys_req_block = soup.find("div", {"data-os": "win"})
    sys_text = sys_req_block.get_text() if sys_req_block else ""
    ram = re.search(r'(\d+)\s*GB\s*RAM', sys_text)
    f['Min_RAM_GB'] = float(ram.group(1)) if ram else 0.0
    storage = re.search(r'(\d+)\s*GB\s*available', sys_text)
    f['Storage_GB'] = float(storage.group(1)) if storage else 0.0

    # Price & Monetization
    price_div = soup.select_one(".game_purchase_price.price")
    price_text = price_div.get_text().lower() if price_div else ""
    f['is_free_to_play'] = "free" in price_text
    p_nums = re.findall(r'\d+', price_text.replace(',', ''))
    f['Price_NTD'] = float(p_nums[0]) if p_nums and not f['is_free_to_play'] else 0.0

    # Steam Ecosystem Features
    feat_text = soup.select_one(".game_area_features_list_ctn").get_text() if soup.select_one(".game_area_features_list_ctn") else ""
    f['feat_multiplayer'] = "Multiplayer" in feat_text
    f['feat_workshop'] = "Workshop" in feat_text
    f['feat_in_app_purchases'] = "In-App Purchases" in feat_text
    f['feat_trading_cards'] = "Trading Cards" in feat_text
    f['feat_achievements'] = "Achievements" in feat_text or soup.find(id="achievement_block") is not None
    f['feat_remote_play'] = "Remote Play" in feat_text

    # Content Counts & Metadata
    f['count_dlcs'] = float(len(soup.select(".game_area_dlc_row")))
    f['count_tags'] = float(len([t for t in soup.select(".app_tag") if "add_button" not in t.get('class', [])]))
    f['is_award_winner'] = soup.select_one(".steamawards_app_banner_event") is not None
    f['is_mature'] = soup.find(id="game_area_content_descriptors") is not None

    return f

# STEP 1: COLLECT APP IDS
all_appids = []
print("Starting AppID collection from SteamSpy...")
for page in range(0, PAGES_TO_FETCH):
    try:
        url = f"https://steamspy.com/api.php?request=all&page={page}"
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            all_appids.extend(list(response.json().keys()))
            print(f"Page {page} collected. Total IDs: {len(all_appids)}")
        time.sleep(1.5)
    except Exception as e:
        print(f"Error on SteamSpy Page {page}: {e}")

# STEP 2: SCRAPE STEAM STORE
final_data = []
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9'
}
# Cookie to bypass birthday check (Age Gate)
cookies = {'birthtime': '283993201', 'lastagecheckage': '1-0-1991'}

print(f"\nStarting extraction for up to {TARGET_COUNT} games...")

for i, appid in enumerate(all_appids[:TARGET_COUNT]):
    url = f"https://store.steampowered.com/app/{appid}/"
    try:
        res = requests.get(url, headers=headers, cookies=cookies, timeout=10)
        
        # Check if redirect to the age check despite cookies
        if "agecheck" in res.url:
            print(f"[{i+1}] Skipped: {url} (Hard Age Gate)")
            continue

        if res.status_code == 200:
            soup = BeautifulSoup(res.text, "html.parser")
            game_data = extract_steam_features_full(soup)
            
            if game_data['Title'] == "Unknown":
                print(f"[{i+1}] Failed to parse: {url}")
                continue
                
            game_data['AppID'] = appid
            game_data['URL'] = url
            final_data.append(game_data)
            print(f"[{i+1}] Processed: {game_data['Title']}")
        
        # Polite delay to prevent IP ban
        time.sleep(random.uniform(1.2, 2.5))
        
        # Intermediate Save every 100 items so you don't lose data
        if (i + 1) % 100 == 0:
            pd.DataFrame(final_data).to_excel(OUTPUT_FILE, index=False)
            print(f"--- Backup saved at {i+1} items ---")

    except Exception as e:
        print(f"[{i+1}] Error on {url}: {e}")

# STEP 3: FINAL PROCESSING
df = pd.DataFrame(final_data)
df.to_excel(OUTPUT_FILE, index=False)

print("\n" + "="*30)
print(f"FINAL DATASET SIZE: {df.shape}")
print(f"SAVED TO: {OUTPUT_FILE}")
print("="*30)

Starting AppID collection from SteamSpy...
Page 0 collected. Total IDs: 1000
Page 1 collected. Total IDs: 2000
Page 2 collected. Total IDs: 3000
Page 3 collected. Total IDs: 4000
Page 4 collected. Total IDs: 5000
Page 5 collected. Total IDs: 6000
Page 6 collected. Total IDs: 7000
Page 7 collected. Total IDs: 8000
Page 8 collected. Total IDs: 9000
Page 9 collected. Total IDs: 10000

Starting extraction for up to 10000 games...
[1] Processed: Counter-Strike 2
[2] Processed: Apex Legends™
[3] Processed: PUBG: BATTLEGROUNDS
[4] Processed: Palworld
[5] Processed: Team Fortress 2
[6] Processed: Call of Duty®
[7] Processed: New World: Aeternum
[8] Processed: Black Myth: Wukong
[9] Processed: Grand Theft Auto V Legacy
[10] Processed: Left 4 Dead 2
[11] Processed: Unturned
[12] Failed to parse: https://store.steampowered.com/app/1599340/
[13] Processed: War Thunder
[14] Processed: Monster Hunter Wilds
[15] Processed: HELLDIVERS™ 2
[16] Processed: Warframe
[17] Processed: ELDEN RING
[18] Process